## Temperature

温度参数 $T$ 用于缩放模型的 Logits 输出，控制概率分布的“陡峭”程度：$$P(x_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$
- $T \to 0$（极低温度）： 最高的 Logits 占据绝对优势，概率分布极度集中，等同于贪婪搜索（Greedy Search）。输出高度确定、严谨，但可能陷入死循环。
- $T = 1.0$： 保留模型原始预训练学到的概率分布。    
- $T > 1.0$（高温度）： 拉平概率分布，缩小高频词与低频词之间的概率差距。生成内容更有创意、随机，但容易引发逻辑混乱和胡言乱语（幻觉）。  

## Top-K Sampling

仅保留预测概率最高的前 $K$ 个 Token，将其余 Token 的 Logits 强行设为 $-\infty$ 后重新归一化。

- 优点： 简单有效，彻底切断长尾的低概率噪声。
- 缺点： 静态的 $K$ 值不够灵活。当模型很确定时（如回答 1 + 1 = ），前 1 个 Token 就够了；当模型需要发散时（如写诗），$K=50$ 都不一定够。

## Top-P Sampling

按概率从高到低排序，动态累加候选 Token 的概率，直到累加概率达到阈值 $p$（如 $p=0.9$），仅保留该集合内的 Token。

- 优点： 动态适应概率分布。如果模型对下一个词非常自信（前两个词概率占 95%），候选池就只包含这 2 个词；如果模型不确定，候选池会自动扩大。
- 缺点： 静态的 $p$ 值不够灵活。当模型很确定时（如回答 1 + 1 = ），$p=0.9$ 就够了；当模型需要发散时（如写诗），$p=0.9$ 都不一定够。 


In [1]:
# ============================================================
# 模块 1：导入依赖 + MiniGPT 模型定义
# 说明：直接从 Mini-GPT.ipynb 搬过来，供 generate.ipynb 独立运行
# ============================================================

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int = 256
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 256
    dropout: float = 0.1

    def __post_init__(self):
        if self.n_embd % self.n_head != 0:
            raise ValueError("n_embd 必须能够被 n_head 整除")


class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head

        self.qkv_proj = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.out_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        causal_mask = torch.tril(torch.ones(config.block_size, config.block_size))
        self.register_buffer(
            "causal_mask",
            causal_mask.view(1, 1, config.block_size, config.block_size),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape                                          # [B, T, C]
        qkv = self.qkv_proj(x)                                     # [B, T, 3C]
        q, k, v = qkv.chunk(3, dim=-1)                             # 各 [B, T, C]

        # [B, T, C] -> [B, H, T, head_dim]
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)  # [B, H, T, T]
        scores = scores.masked_fill(
            self.causal_mask[:, :, :T, :T] == 0,
            torch.finfo(scores.dtype).min,
        )
        attention = self.attn_dropout(F.softmax(scores, dim=-1))     # [B, H, T, T]
        y = attention @ v                                             # [B, H, T, head_dim]

        # 合并多头: [B, H, T, head_dim] -> [B, T, C]
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(y))


class MLP(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)                                           # [B, T, C] -> [B, T, C]


class GPTBlock(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attention = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attention(self.ln1(x))                          # Pre-LN + 残差
        x = x + self.mlp(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(config.block_size, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([GPTBlock(config) for _ in range(config.n_layer)])
        self.final_norm = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.lm_head.weight = self.token_embedding.weight            # 权重共享
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module: nn.Module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, labels: torch.Tensor | None = None):
        B, T = input_ids.shape                                      # [B, T]
        assert T <= self.config.block_size, \
            f"序列长度 {T} 超过最大 {self.config.block_size}"

        positions = torch.arange(T, device=input_ids.device)        # [T]
        token_emb = self.token_embedding(input_ids)                 # [B, T, C]
        position_emb = self.position_embedding(positions)           # [T, C]
        x = self.dropout(token_emb + position_emb)                   # [B, T, C]

        for block in self.blocks:
            x = block(x)                                             # [B, T, C]

        x = self.final_norm(x)                                       # [B, T, C]
        logits = self.lm_head(x)                                     # [B, T, V]

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()            # [B, T-1, V]
            shift_labels = labels[:, 1:].contiguous()                # [B, T-1]
            loss = F.cross_entropy(
                shift_logits.view(-1, self.config.vocab_size),       # [B*(T-1), V]
                shift_labels.view(-1),                               # [B*(T-1)]
                ignore_index=-100,
            )
        return logits, loss

print("✅ 模型定义加载完成")

✅ 模型定义加载完成


In [2]:
# ============================================================
# 模块 2：字符级分词器
# 每个唯一字符映射为一个 token ID，简单直观
# ============================================================

class CharTokenizer:
    def __init__(self, text: str):
        chars = sorted(list(set(text)))                              # 去重排序
        self.stoi = {ch: i for i, ch in enumerate(chars)}            # 字符 -> ID
        self.itos = {i: ch for i, ch in enumerate(chars)}            # ID -> 字符
        self.vocab_size = len(chars)

    def encode(self, s: str) -> list[int]:
        return [self.stoi[ch] for ch in s]

    def decode(self, ids: list[int]) -> str:
        return "".join(self.itos[i] for i in ids)

    def __repr__(self):
        return f"CharTokenizer(vocab_size={self.vocab_size})"

print("✅ 分词器定义完成")

✅ 分词器定义完成


In [3]:
# ============================================================
# 模块 3：数据加载
# 读取文本 -> 编码为一维张量 -> 随机采样 batch
# ============================================================

def load_text(filepath: str):
    """读取文本文件并返回内容"""
    with open(filepath, "r", encoding="utf-8") as f:
        return f.read()


def get_batch(data: torch.Tensor, batch_size: int, block_size: int):
    """
    从一维序列中随机采样 batch
    data: [N] 整条语料编码后的一维张量
    返回: x [B, T], y [B, T]（y 比 x 向后位移一位）
    """
    ix = torch.randint(len(data) - block_size, (batch_size,))          # [B] 采样起始索引
    x = torch.stack([data[i : i + block_size]          for i in ix])   # [B, T]
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])   # [B, T]
    return x, y


def split_data(data: torch.Tensor, split_ratio: float = 0.9):
    """按比例划分训练集和验证集"""
    n = int(len(data) * split_ratio)
    return data[:n], data[n:]

print("✅ 数据加载函数定义完成")

✅ 数据加载函数定义完成


In [4]:
# ============================================================
# 模块 4：训练循环
# 包含训练和周期性评估，定期打印 loss
# ============================================================

def train(
    model: nn.Module,
    train_data: torch.Tensor,
    val_data: torch.Tensor,
    batch_size: int = 32,
    max_steps: int = 5000,
    eval_interval: int = 500,
    eval_iters: int = 200,
    lr: float = 3e-4,
    device: str = "cpu",
):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    @torch.no_grad()
    def estimate_loss():
        """在训练集和验证集上各采样 eval_iters 次，计算平均 loss"""
        model.eval()
        losses = {}
        for name, data in [("train", train_data), ("val", val_data)]:
            total = 0.0
            for _ in range(eval_iters):
                x, y = get_batch(data, batch_size, model.config.block_size)
                x, y = x.to(device), y.to(device)
                _, loss = model(x, labels=y)
                total += loss.item()
            losses[name] = total / eval_iters
        model.train()
        return losses

    for step in range(1, max_steps + 1):
        # 周期性评估
        if step % eval_interval == 0 or step == 1:
            losses = estimate_loss()
            print(
                f"step {step:>5d}/{max_steps}  "
                f"train_loss: {losses['train']:.4f}  "
                f"val_loss: {losses['val']:.4f}"
            )

        # 单步训练
        x, y = get_batch(train_data, batch_size, model.config.block_size)
        x, y = x.to(device), y.to(device)
        logits, loss = model(x, labels=y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("✅ 训练完成")
    return model

print("✅ 训练函数定义完成")

✅ 训练函数定义完成


In [5]:
# ============================================================
# 模块 5：文本生成（自回归推理）
# 给定 prompt，模型逐个预测下一个 token，然后拼回
# ============================================================

@torch.no_grad()
def generate(
    model: MiniGPT,
    tokenizer: CharTokenizer,
    prompt: str,
    max_new_tokens: int = 200,
    temperature: float = 1.0,
    device: str = "cpu",
) -> str:
    model.eval()
    ids = tokenizer.encode(prompt)                                    # list[int]
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)  # [1, T_prompt]

    for _ in range(max_new_tokens):
        # 截断：如果长度超过 block_size，只保留最后 block_size 个 token
        input_cond = input_ids[:, -model.config.block_size:]          # [1, min(T, block_size)]
        logits, _ = model(input_cond)                                 # [1, T_cond, V]
        logits = logits[:, -1, :]                                     # [1, V] 取最后一个位置

        # 温度缩放：temperature 越低越确定，越高越随机
        logits = logits / temperature
        probs = F.softmax(logits, dim=-1)                             # [1, V]
        next_id = torch.multinomial(probs, num_samples=1)             # [1, 1]
        input_ids = torch.cat([input_ids, next_id], dim=1)            # [1, T+1]

    return tokenizer.decode(input_ids[0].tolist())

print("✅ 生成函数定义完成")

✅ 生成函数定义完成


In [ ]:
@torch.no_grad()
def generate_optimize(
    model: MiniGPT,
    tokenizer: CharTokenizer,
    prompt: str,
    max_new_tokens: int = 200,
    temperature: float = 1.0,
    top_k: int = 0,            # 0 表示不启用 Top-K 过滤
    top_p: float = 1.0,          # 1.0 表示不启用 Top-P 过滤
    device: str = "cpu",
) -> str:
    model.eval()
    ids = tokenizer.encode(prompt)                                    # list[int]
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)  # [1, T_prompt]

    for _ in range(max_new_tokens):
        # 1. 截断：如果长度超过 block_size，只保留最后 block_size 个 token
        input_cond = input_ids[:, -model.config.block_size:]          # [1, min(T, block_size)]
        logits, _ = model(input_cond)                                 # [1, T_cond, V]
        logits = logits[:, -1, :]                                     # [1, V] 取最后一个位置

        # 2. 温度缩放 (Temperature Scaling)
        if temperature > 0:
            logits = logits / temperature

        # 3. Top-K 截断（保留得分最高的前 K 个 Token）
        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            # 将低于第 K 个最高得分的位置设为 -inf
            logits[logits < v[:, [-1]]] = -float('Inf')

        # 4. Top-P (Nucleus) 截断（保留累积概率达到 top_p 的前若干个 Token）
        if top_p < 1.0 and top_p > 0.0:
            # 降序排列 logits
            sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
            # 计算 Softmax 后的累积概率
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

            # 标记累积概率超过 top_p 的位置
            sorted_indices_to_remove = cumulative_probs > top_p
            # 向右平移 1 位，确保包含刚好超过 top_p 阈值的那个 Token
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            # 将掩码还原回原始 logits 词表顺序
            indices_to_remove = sorted_indices_to_remove.scatter(
                dim=-1, index=sorted_indices, src=sorted_indices_to_remove
            )
            logits[indices_to_remove] = -float('Inf')

        # 5. 计算概率并采样
        probs = F.softmax(logits, dim=-1)                             # [1, V]
        next_id = torch.multinomial(probs, num_samples=1)             # [1, 1]
        input_ids = torch.cat([input_ids, next_id], dim=1)            # [1, T+1]

    return tokenizer.decode(input_ids[0].tolist())

In [6]:
# ============================================================
# 模块 6：主流程
# 选择训练语料，一键完成「分词 → 训练 → 生成」
# ============================================================

# ---------- 超参数 ----------
TEXT_FILE = "../data/timemachine.txt"         # 训练文本文件路径（放在当前目录下）
BATCH_SIZE = 32
MAX_STEPS = 5000
EVAL_INTERVAL = 500
LEARNING_RATE = 3e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---------- 1. 读取文本 & 分词 ----------
print(f"📖 正在读取: {TEXT_FILE}")
raw_text = load_text(TEXT_FILE)
print(f"   文本长度: {len(raw_text)} 字符")

tokenizer = CharTokenizer(raw_text)
print(f"   词表大小: {tokenizer.vocab_size}")

data = torch.tensor(tokenizer.encode(raw_text), dtype=torch.long)     # [N]
train_data, val_data = split_data(data, split_ratio=0.9)
print(f"   训练集: {len(train_data)} tokens, 验证集: {len(val_data)} tokens")

# ---------- 2. 创建模型 ----------
model_config = GPTConfig(
    vocab_size=tokenizer.vocab_size,
    block_size=128,
    n_layer=4,
    n_head=4,
    n_embd=256,
    dropout=0.1,
)
model = MiniGPT(model_config)
total_params = sum(p.numel() for p in model.parameters())
print(f"   模型参数量: {total_params / 1e6:.2f}M")

# ---------- 3. 训练 ----------
print(f"\n🚀 开始训练（设备: {DEVICE}）")
print("-" * 50)
model = train(
    model=model,
    train_data=train_data,
    val_data=val_data,
    batch_size=BATCH_SIZE,
    max_steps=MAX_STEPS,
    eval_interval=EVAL_INTERVAL,
    lr=LEARNING_RATE,
    device=DEVICE,
)

# ---------- 4. 生成 ----------
print("\n" + "=" * 50)
print("📝 文本生成示例")
print("=" * 50)

prompts = [
    "time machine:",                           
    "time:",                                   
    raw_text[:3],                               # 用前几个字符作为 prompt
]

for prompt in prompts:
    print(f"\n--- Prompt: {prompt!r} ---")
    generated = generate(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=200,
        temperature=0.6,
        device=DEVICE,
    )
    print(generated[:300])                      # 打印前 300 字符
    print("...")

📖 正在读取: ../data/timemachine.txt
   文本长度: 178979 字符
   词表大小: 70
   训练集: 161081 tokens, 验证集: 17898 tokens
   模型参数量: 3.21M

🚀 开始训练（设备: cuda）
--------------------------------------------------
step     1/5000  train_loss: 4.3265  val_loss: 4.3208
step   500/5000  train_loss: 2.6031  val_loss: 2.6272
step  1000/5000  train_loss: 2.2397  val_loss: 2.3690
step  1500/5000  train_loss: 1.9718  val_loss: 2.2155
step  2000/5000  train_loss: 1.7463  val_loss: 2.1970
step  2500/5000  train_loss: 1.5108  val_loss: 2.2373
step  3000/5000  train_loss: 1.2846  val_loss: 2.3269
step  3500/5000  train_loss: 1.0785  val_loss: 2.4291
step  4000/5000  train_loss: 0.8849  val_loss: 2.5817
step  4500/5000  train_loss: 0.7287  val_loss: 2.7058
step  5000/5000  train_loss: 0.6048  val_loss: 2.8431
✅ 训练完成

📝 文本生成示例

--- Prompt: 'time machine:' ---
time machine:smahnso
o aog,ca hwfoee feligtns frml nguse o hll uoe yul,adatl hstedb oe rie nmsfr,adha
eas rme rtd nt ntr pas hslp.Te n ububuh h h h    iettgpl h fltafc